# Uniform Morton-sorted tree geometry

## Purpose

The `UniformTree` organises source and target geometry into a complete,
non-adaptive octree. Nodes are stored level by level and in Morton order within
each level. This notebook inspects box geometry, source permutations, leaf
occupancy, and the near (`list1`) and well-separated (`list2`) interaction
lists.

**Current scope:** the tree supplies geometry and lists only. It does not run
an FMM upward pass, M2L traversal, downward pass, or particle evaluation.

## Morton ordering and interaction lists

At level $\ell$, integer box coordinates each contain $\ell$ bits. Morton
ordering interleaves the x, y, and z bits so spatially nearby boxes tend to
remain nearby in linear storage.

- `list1` contains touching same-level boxes, including the selected box.
- `list2` contains children of the parent neighbourhood that are not in
  `list1`; these boxes are well separated at the current level but their
  parents touch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
n_sources = 120
random_seed = 42
max_level = 2
displayed_level = 2
selected_level = 2
selected_coordinates = (1, 1, 1)

## Build and summarise the tree

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = rng.normal(size=(n_sources, 3))
source_positions[:, 0] *= 1.4
source_positions[:, 2] *= 0.7

options = cdfmm.UniformTreeOptions()
options.max_level = max_level
tree = cdfmm.UniformTree(source_positions, options)

root_centre = vec3_to_array(tree.root_centre)
leaf_nodes = nodes_at_level(tree, tree.leaf_level)
print(f"Root centre: {root_centre}")
print(f"Root half-width: {tree.root_half_width:.6f}")
print(f"Number of levels: {tree.n_levels}")
print(f"Total node count: {len(tree.nodes)}")
print(f"Leaf count: {len(leaf_nodes)}")
print(f"Source permutation (first 15): {tree.source_permutation[:15]}")
print("Sorted source positions (first five):")
print(tree.sorted_source_positions()[:5])

## A. Source distribution and actual tree boxes

In [ ]:
figure, axes = new_3d_figure(figsize=(9, 8))
axes.scatter(*source_positions.T, s=12, color="tab:blue", alpha=0.7, label="sources")

for node in nodes_at_level(tree, displayed_level):
    draw_box_3d(
        axes,
        vec3_to_array(node.centre),
        node.half_width,
        colour="0.35",
        linewidth=0.6,
        alpha=0.55,
    )

finish_3d_axes(axes, f"Complete uniform-tree boxes at level {displayed_level}")
axes.legend()
figure.tight_layout()

## B. Input order versus Morton-sorted order

In [ ]:
sorted_positions = tree.sorted_source_positions()
sorted_leaf_morton = np.array(
    [
        tree.nodes[tree.leaf_index_for_source(sorted_index)].morton_index
        for sorted_index in range(n_sources)
    ]
)
original_leaf_morton = np.empty(n_sources, dtype=int)
original_leaf_morton[np.asarray(tree.source_permutation)] = sorted_leaf_morton

figure, axes = plt.subplots(1, 2, figsize=(13, 5))
input_scatter = axes[0].scatter(
    source_positions[:, 0],
    source_positions[:, 1],
    c=original_leaf_morton,
    cmap="viridis",
    s=28,
)
axes[0].set_title("Input order, coloured by leaf Morton index")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_aspect("equal")

axes[1].plot(sorted_leaf_morton, ".-", color="tab:purple")
axes[1].set_title("Non-decreasing Morton index in sorted storage")
axes[1].set_xlabel("Sorted source index")
axes[1].set_ylabel("Leaf Morton index")
figure.colorbar(input_scatter, ax=axes[0], label="Leaf Morton index")
figure.tight_layout()

## C. Leaf occupancy

In [ ]:
leaf_morton_indices = np.array([node.morton_index for node in leaf_nodes])
leaf_occupancies = np.array([node.source_count for node in leaf_nodes])

figure, axes = plt.subplots(figsize=(11, 4.5))
axes.bar(leaf_morton_indices, leaf_occupancies, color="tab:blue")
axes.set_xlabel("Leaf Morton index")
axes.set_ylabel("Number of sources")
axes.set_title(f"Source occupancy across {len(leaf_nodes)} complete-tree leaves")
figure.tight_layout()

## D. Visualise list1 and list2

In [ ]:
selected_node = next(
    node
    for node in nodes_at_level(tree, selected_level)
    if (node.ix, node.iy, node.iz) == selected_coordinates
)

figure, axes = new_3d_figure(figsize=(9, 8))

# Draw all same-level boxes lightly to retain geometric context.
for node in nodes_at_level(tree, selected_level):
    draw_box_3d(
        axes,
        vec3_to_array(node.centre),
        node.half_width,
        colour="0.82",
        linewidth=0.45,
        alpha=0.35,
    )

for list_position, node_index in enumerate(selected_node.list2):
    node = tree.nodes[node_index]
    draw_box_3d(
        axes,
        vec3_to_array(node.centre),
        node.half_width,
        colour="tab:orange",
        linewidth=1.15,
        alpha=0.9,
        label="list2: well-separated" if list_position == 0 else None,
    )

for list_position, node_index in enumerate(selected_node.list1):
    node = tree.nodes[node_index]
    draw_box_3d(
        axes,
        vec3_to_array(node.centre),
        node.half_width,
        colour="tab:blue",
        linewidth=1.4,
        alpha=0.95,
        label="list1: touching" if list_position == 0 else None,
    )

draw_box_3d(
    axes,
    vec3_to_array(selected_node.centre),
    selected_node.half_width,
    colour="tab:red",
    linewidth=2.8,
    alpha=1.0,
    label="selected node",
)
finish_3d_axes(
    axes,
    f"Interaction lists for level {selected_level}, Morton {selected_node.morton_index}",
)
axes.legend(loc="upper left")
figure.tight_layout()

print(f"Selected flat node index: {selected_node.index}")
print(f"Selected integer coordinates: {(selected_node.ix, selected_node.iy, selected_node.iz)}")
print(f"list1 box count: {len(selected_node.list1)}")
print(f"list2 box count: {len(selected_node.list2)}")

## E. Interactive tree inspection

In [ ]:
from ipywidgets import IntSlider, interact


def inspect_tree(max_level=2, displayed_level=2, selected_node=0):
    dynamic_options = cdfmm.UniformTreeOptions()
    dynamic_options.max_level = max_level
    dynamic_tree = cdfmm.UniformTree(source_positions, dynamic_options)
    level = min(displayed_level, max_level)
    level_nodes = nodes_at_level(dynamic_tree, level)
    selected_position = min(selected_node, len(level_nodes) - 1)
    selected = level_nodes[selected_position]

    figure, axes = new_3d_figure(figsize=(7, 6))
    for node in level_nodes:
        colour = "tab:red" if node.index == selected.index else "0.65"
        linewidth = 2.2 if node.index == selected.index else 0.5
        draw_box_3d(
            axes,
            vec3_to_array(node.centre),
            node.half_width,
            colour=colour,
            linewidth=linewidth,
            alpha=0.75,
        )
    axes.scatter(*source_positions.T, s=6, color="tab:blue", alpha=0.35)
    finish_3d_axes(
        axes,
        f"Level {level}; selected Morton index {selected.morton_index}",
    )
    figure.tight_layout()
    plt.show()
    print(
        f"nodes={len(dynamic_tree.nodes)}, level boxes={len(level_nodes)}, "
        f"list1={len(selected.list1)}, list2={len(selected.list2)}"
    )


interact(
    inspect_tree,
    max_level=IntSlider(min=1, max=3, step=1, value=2),
    displayed_level=IntSlider(min=0, max=3, step=1, value=2),
    selected_node=IntSlider(min=0, max=511, step=1, value=0),
)

## What to observe

Every level is complete, including empty boxes, and sources become contiguous
after sorting by their leaf Morton index. `list1` forms the touching
neighbourhood around the selected node. `list2` surrounds that neighbourhood
with boxes whose parents are neighbours, making the distinction between near
and admissible far interactions visually explicit. No numerical FMM pass is
performed here.